In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col, current_timestamp

spark = SparkSession.builder \
    .appName("bronze-layer") \
    .config("spark.jars.packages",
            "org.postgresql:postgresql:42.6.0,"
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.lakehouse.type", "hadoop") \
    .config("spark.sql.catalog.lakehouse.warehouse", "/tmp/warehouse") \
    .getOrCreate()

print("✅ Spark com Iceberg iniciado!")

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.bronze")

In [ ]:
df_raw = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/pipeline_transacoes") \
    .option("dbtable", "staging.transacoes_raw") \
    .option("user", "postgres") \
    .option("password", "1234") \
    .load()

df_raw.show(5)

In [ ]:
df_bronze = df_raw.withColumn(
    "data_ingestao",
    current_timestamp()
).withColumn(
    "data_particao",
    to_date(col("data_ingestao"))
)

df_bronze.show(5)

In [ ]:
df_bronze.writeTo("lakehouse.bronze.transacoes") \
    .partitionedBy("data_particao") \
    .createOrReplace()

print("✅ Dados gravados na Bronze")

In [ ]:
df_bronze.writeTo("lakehouse.bronze.transacoes") \
    .partitionedBy("data_particao") \
    .createOrReplace()

print("✅ Dados gravados na Bronze")

In [ ]:

spark.sql("SHOW PARTITIONS lakehouse.bronze.transacoes").show()

In [ ]:
!ls /tmp/warehouse/bronze/transacoes

In [ ]:
!ls /tmp/warehouse/bronze/transacoes